# EXIST 2026 — Task 3 (vídeos) — braço CLIP (item 10)

**⚠ BRAÇO DE ATRIBUIÇÃO ISOLADO.** Este notebook treina uma variante do modelo de vídeo
trocando o encoder visual de VideoMAE-base (pré-treinado em vídeo, LoRA) por
CLIP-ViT-L/14 (pré-treinado em imagem, 4 últimas camadas destravadas full-rank — mesmo
padrão de `EXIST_MEMES/memes_core.py:488-494`). Motivação e decisão em
`improvements/16_VIDEO_INVESTIGATION_RESULTS.md` (H2: o ramo VideoMAE é fraco por
capacidade absoluta + objetivo de pré-treino, não por defeito de pooling/adaptação) e
`improvements/17_RETRAIN_IMPROVEMENT_OPTIONS.md` (CLIP como intervenção de maior teto).

**Protocolo de seleção de estado (decisão final, pós-investigação):** este braço é
reportado no **estado FINAL da época 15**, não em `E=argmin(eval_loss)`. Duas runs
anteriores testaram a hipótese de overtraining (E=argmin cedo, época 2-4) e a hipótese
de desequilíbrio de capacidade do encoder (`UNFROZEN_VISION_LAYERS` 4→2) — ambas
refutadas pela sonda causal (o slot de texto não recuperou; o de vídeo, se algo,
dominou mais). O ramo de texto ficar quase constante na fusão é um trade-off real do
CLIP substituir VideoMAE, não um artefato de overtraining a ser corrigido por seleção
de época. Regra de simetria do projeto (CLAUDE.md): estado final vs estado final, nunca
fim de um lado e pico do outro — e o baseline VideoMAE (B15) também é reportado no seu
estado final. `save_strategy`/callbacks de época (seção 9/10) continuam gravando o
histórico por época **só como registro diagnóstico**; nada dali é recarregado por cima
do modelo final (ver seção 12).

**Não substitui o baseline VideoMAE.** Este é um braço de ablação de arquitetura,
paralelo a `Thesis_video_improved.ipynb`, não um sucessor dele:

- **Incomparável por `state_dict`** com as runs VideoMAE (B15, A, A′, CAL, baseline
  congelado) — encoder visual diferente, dimensão de vídeo diferente (1024 vs 768),
  largura de fusão diferente (2944/2816 vs 2688/2560), grupo de LR novo no otimizador.
  `meta.json["video_encoder"]` é o desambiguador autoritativo; nunca inferir o tipo de
  encoder pelas chaves do `state_dict` (elas nascem como `videomae.clip.*` — wart
  documentada, ver `improvements/18_CLIP_VIDEO_BRANCH_SPEC.md`).
- **Divergência de interpolação conhecida e não resolvida aqui:** os frames em disco
  foram extraídos com o resize/normalização do VideoMAE (bilinear, ImageNet). Este
  notebook renormaliza aritmeticamente para a estatística do CLIP (afim composta,
  `EXISTVideoDatasetCLIP`), mas **não re-interpola** — o `CLIPImageProcessor` padrão
  usa bicubic. Essa divergência bilinear(VideoMAE)/bicubic(CLIP) é uma limitação
  conhecida e declarada, registrada em `improvements/18_CLIP_VIDEO_BRANCH_SPEC.md`
  (escrito em paralelo a este notebook — ver esse arquivo para o detalhe, não repetido
  aqui).
- **Achado de robustez a declarar na tese, não a corrigir aqui:** a sonda causal mostra
  o slot de vídeo carregando informação de "registro/formato da fonte" (classe de B),
  não necessariamente conteúdo. Um modelo que troca o canal de conteúdo (transcrição)
  por um canal de formato pode generalizar pior no teste real mesmo com métricas de
  validação melhores — argumento de robustez contra este braço, independente do ganho
  agregado. Ver `improvements/16_VIDEO_INVESTIGATION_RESULTS.md`.
- Toda a lógica nova (dataset, encoder, modelo, otimizador, save/load, selfchecks) vive
  em `video_core_clip.py`, que importa de `video_core.py` sem duplicar — `video_core.py`
  e `Thesis_video_improved.ipynb` ficam **intocados**.

Este notebook, como o `Thesis_video_improved.ipynb` que o molda, **não define lógica**:
é orquestração pura (carregar dados, split congelado, scaler, instanciar, treinar,
avaliar, salvar).


## 1. Imports

Nada de lógica aqui — só os módulos e o mínimo para orquestrar. `video_core` fornece o
split congelado, o dataset base, o `Trainer` multi-task e as métricas; `video_core_clip`
fornece só o que muda para o encoder CLIP (dataset renormalizado, modelo, otimizador,
save/load, selfchecks).

In [ ]:
# P1-fix: os caminhos deste notebook sao RELATIVOS a EXIST_VIDEOS/ (plano 11 §3:
# "kernel do Jupyter em /home/a67606/Thesis/EXIST_VIDEOS ou cd ate la"). Se o kernel
# nascer noutro cwd, o assert do JSON da celula seguinte falha. Idempotente.
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretorio do projeto nao encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
print("cwd:", Path.cwd())


In [ ]:
import json

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import RobustScaler
from transformers import TrainingArguments, XLMRobertaTokenizer

import video_core as vc
import video_core_clip as vcc


## 2. Configuração da run

**Braço único: `ARM = "CLIP"`.** Diferente de `Thesis_video_improved.ipynb` (que
seleciona entre CAL/A_prime/A/B/B15 na mesma família VideoMAE), este notebook só tem um
braço — a substituição de encoder é a variável em teste, não a atribuição de features.

- `SPLIT_SEED` é **fixo** e vive em `video_core` — o mesmo split congelado de todas as
  outras runs.
- `INIT_SEED` = 42 por padrão (única seed rodada para este braço de atribuição isolado;
  não faz parte da tabela de ablação cross-seed).

In [ ]:
INIT_SEED      = 42        # seed única para este braço de atribuição isolado
USE_BIO        = True      # False -> fusion_input_dim_clip(False) = 2816
USE_VIEWER_EMB = True      # segue o mesmo padrão do sistema VideoMAE

CONDITION = vc.condition_tag(USE_BIO, USE_VIEWER_EMB)

# NUM_EPOCHS, RUN_DIR e CKPT_DIR sao definidos na secao 2.1 (ARM="CLIP"), que e a
# UNICA fonte deles. Nao redefinir aqui.
print(f"condição = {CONDITION} | INIT_SEED = {INIT_SEED} | SPLIT_SEED = {vc.SPLIT_SEED}")
print(f"max_length = {vc.MAX_LENGTH}")


## 2.1 Braço CLIP (item 10)

Diferente da atribuição em 3 braços do notebook VideoMAE (A′/A/B, que isola ARCH-01 e
áudio), este braço isola **a troca de encoder visual**. Áudio v2 (já corrigido, ARCH-04)
e `restore_biases` **não se aplicam** aqui — não há VideoMAE neste caminho, logo não há
`q_bias`/`v_bias` a restaurar.

**Protocolo de épocas (decisão final):** roda com o mesmo orçamento de 15 épocas do
baseline congelado (`vc.CALIBRATION_EPOCHS`) por comparabilidade de custo com B15, e é
reportado no **estado FINAL da época 15** — não `E=argmin(eval_loss)`. Duas runs
anteriores exploraram seleção por argmin (hipótese de overtraining) e redução de
capacidade do encoder (hipótese de desequilíbrio) como possíveis correções para o
ramo de texto ficar quase constante na fusão; ambas refutadas pela sonda causal
(`improvements/16_VIDEO_INVESTIGATION_RESULTS.md`). O padrão é lido agora como
trade-off do braço, não bug — reportado no estado final por simetria com o baseline
VideoMAE (também reportado no seu estado final, regra do CLAUDE.md).

**⚠ Estrutura de comparação desta run (`ARM="CLIP_WC42"`):** este notebook agora treina
o retreino com a Task C repesada (`LOSS_W_C` 0,7→4,2, opção 5 do doc 20 — ver seção 2.2).
Logo:

- **WC42 vs `run_..._CLIP`** = a repesagem ISOLADA (mesmo encoder, mesmas 4 camadas,
  mesmas seeds, mesmo estado final da época 15). É esta a comparação controlada.
- **WC42 vs B15** = encoder + repesagem, **dois eixos** — leitura qualitativa de teto,
  nunca comparação controlada (a mesma ressalva que já valia para `CLIP` vs B15).

⚠ Frames em disco são v2 (únicos existentes — a árvore v1 foi apagada, V6). Áudio v2
(mesma escolha do braço B/B15 do notebook VideoMAE — é o áudio corrigido, ARCH-04).

In [ ]:
from pathlib import Path

# ARM="CLIP_WC42": retreino do braço CLIP com a Task C repesada (LOSS_W_C 0,7 -> 4,2,
# opção 5 do doc 20). Tudo o mais CONGELADO em relação ao braço "CLIP": 4 camadas
# destravadas, estado FINAL da época 15 (não argmin), mesmas seeds, mesma
# TrainingArguments. "CLIP" (baseline deste retreino) e "CLIP_L2" (2 camadas, argmin —
# hipótese de desequilíbrio de capacidade, refutada) continuam intactos em disco.
ARM = "CLIP_WC42"

AUDIO_VERSION = "v2"   # áudio v2 (ARCH-04 corrigido), mesma escolha do braço B/B15
NUM_EPOCHS = vc.CALIBRATION_EPOCHS   # 15 — mesma TrainingArguments do baseline p/ comparabilidade de custo

# Sufixo _CLIP_WC42: diretório NOVO, nunca deve colidir com os diretórios VideoMAE
# (sufixos _CAL/_A_prime/_A/_B/_B15) nem com run_..._CLIP, que é o baseline contra o
# qual esta run será comparada e NÃO pode ser sobrescrito.
RUN_DIR  = Path(f"{vc.run_dir(INIT_SEED, USE_BIO, USE_VIEWER_EMB)}_{ARM}")
CKPT_DIR = Path(f"{vc.ckpt_dir(INIT_SEED, USE_BIO, USE_VIEWER_EMB)}_{ARM}")

print(f"ARM = {ARM} | áudio = {AUDIO_VERSION} | frames = v2 | encoder = CLIP-ViT-L/14 "
      f"({vcc.UNFROZEN_VISION_LAYERS} camadas destravadas) | estado reportado = FINAL")
print(f"LOSS_W_C alvo = {vcc.LOSS_W_C_REWEIGHTED} (aplicado na célula seguinte)")
print(f"NUM_EPOCHS = {NUM_EPOCHS}")
print(f"RUN_DIR  = {RUN_DIR}")
print(f"CKPT_DIR = {CKPT_DIR}")


## 2.2 Repesagem da Task C na loss (opção 5 do doc 20)

`loss_task_c` reduz as 6 classes com `.mean(dim=-1)` enquanto `loss_task_b` usa
`.sum(dim=-1)`: com `LOSS_W_B = LOSS_W_C = 0.7`, o peso EFETIVO de C é `0.7 * 1/6`.
`0.7 * 6 = 4.2` desfaz exatamente essa assimetria aritmética — **não** é um valor
afinado em validação (isso seria leakage).

A mudança é feita mutando `vc.LOSS_W_C` em runtime (`vc.compute_loss` resolve o global
a cada chamada), porque `video_core.py` **nunca é editado** neste projeto.

⚠ **A `eval_loss` desta run não é comparável com a das runs anteriores** — o termo de C
entra 6× maior. Só as 6 métricas do PyEvALL comparam.

⚠ Isto é **realocação**, não ganho grátis: subir C deve custar A e/ou B. O relatório
final tem de trazer as seis métricas, não só a que melhorou.


In [ ]:
LOSS_WEIGHTS = vcc.apply_loss_reweight_c()   # LOSS_W_C: 0.7 -> 4.2

assert vc.LOSS_W_C == vcc.LOSS_W_C_REWEIGHTED, (
    f"LOSS_W_C={vc.LOSS_W_C} — a repesagem não pegou")


## 3. Determinismo e self-checks

`set_all_seeds` é chamado de novo imediatamente antes de cada init de pesos — é o que
`vcc.build_model_clip` faz internamente (mesmo protocolo do `vc.build_model`: as
matrizes LoRA A do XLM-R e as camadas destravadas do CLIP nascem de RNG explicitamente
re-semeado, não do `set_seed` tardio do Trainer).

`vc.run_selfchecks()` continua válido aqui — cobre convenções da Task A, taxa de
modality dropout, etc., que não mudam com o encoder de vídeo.

In [ ]:
vc.set_all_seeds(INIT_SEED)
vc.run_selfchecks()


## 4. Dados e split **congelado**

Mesmo split congelado (`Dataset/val_split_42.json`) de todas as outras runs — o braço
CLIP compartilha o mesmo val, para que qualquer comparação futura (mesmo que informal,
já que o braço é incomparável por state_dict) ao menos use os mesmos itens de avaliação.

⚠ **O treino roda numa VM separada e `Dataset/` não é sincronizado.** Copiar
`Dataset/val_split_42.json` para a VM antes da primeira run: a célula abaixo tem de
imprimir `[split] carregado de ...`. Se imprimir `[split] artefato gravado`, a run está
**fora de protocolo**.

In [ ]:
assert vc.split_artifact_path().exists(), (
    "Copiar Dataset/val_split_42.json para a VM antes de rodar. Sem ele, "
    "build_frozen_split RECALCULA o split em vez de carregar o congelado.")

full_df = pd.read_json("./Dataset/training/EXIST2026_training.json").transpose()
train_df, val_df = vc.build_frozen_split(full_df)
print(f"Treino: {len(train_df)} | Val: {len(val_df)}")


## 5. `RobustScaler` das features biométricas

Ajustado **só no treino** (anti-leakage) e gravado no `RUN_DIR`, via `vcc.save_run_clip`.

In [ ]:
train_bio_matrix = vc.collect_bio_vectors(train_df)
scaler = RobustScaler().fit(train_bio_matrix)
print("matriz bio de treino:", train_bio_matrix.shape)


## 6. Datasets

`vcc.EXISTVideoDatasetCLIP` herda de `vc.EXISTVideoDataset` — carrega os mesmos frames
v2 em disco (normalizados para VideoMAE/ImageNet) e renormaliza por afim composta para
a estatística do CLIP, sem reler disco. `max_length=256` (ARCH-05), igual ao braço
VideoMAE.

In [ ]:
tokenizer = XLMRobertaTokenizer.from_pretrained(vc.XLMR_NAME)

train_dataset = vcc.EXISTVideoDatasetCLIP(
    train_df, tokenizer, scaler,
    vc.frames_dir("train"), vc.audio_dir("train", version=AUDIO_VERSION), max_length=vc.MAX_LENGTH)
val_dataset = vcc.EXISTVideoDatasetCLIP(
    val_df, tokenizer, scaler,
    vc.frames_dir("train"), vc.audio_dir("train", version=AUDIO_VERSION), max_length=vc.MAX_LENGTH)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


## 7. Sanity checks dos dados

Além do shape dos frames (idêntico ao braço VideoMAE — o dataset CLIP consome o mesmo
tensor `[16,3,224,224]`, só muda a estatística de normalização), roda-se aqui
`vcc.selfcheck_renorm_equivalence` — verifica sobre itens reais em disco que a
renormalização afim composta (VideoMAE → CLIP) é aritmeticamente equivalente a
renormalizar a partir do zero, dentro de `atol=1e-5`. Portão só executável na VM (com
frames reais em disco); localmente não tem valor probatório.

In [ ]:
sample = train_dataset[0]
for k, v in sample.items():
    print(f"{k:20s}: {tuple(v.shape) if torch.is_tensor(v) else v}")

assert sample["frames"].shape == (vc.NUM_FRAMES, vc.NUM_CHANNELS, vc.FRAME_H, vc.FRAME_W)
print("✓ shape dos frames confere (renormalizados para a estatística do CLIP)")

print("'labels' no batch?", "labels" in sample, "(tem de ser False)")

cvpc_values = [train_dataset[i]["cvpc"].item() for i in range(len(train_dataset))]
print(f"CVPC mean={np.mean(cvpc_values):.4f} std={np.std(cvpc_values):.4f} "
      f"range=[{min(cvpc_values):.4f}, {max(cvpc_values):.4f}]")

# Portão: equivalência da renormalização afim composta contra recomputar do zero.
vcc.selfcheck_renorm_equivalence(train_dataset)
print("✓ selfcheck_renorm_equivalence passou")


## 8. Modelo

`vcc.build_model_clip` faz, no protocolo de determinismo: re-seed → `build_backbones_clip`
(CLIP-ViT-L/14 com as 4 últimas camadas de visão destravadas full-rank, sem LoRA — mesmo
padrão de `EXIST_MEMES/memes_core.py:488-494` — e XLM-R com LoRA via `vc.make_lora_config`)
→ re-seed → `EXISTVideoModelCLIP` na condição pedida.

`fusion_in` esperado: **2944** com bio (`USE_BIO=True`), **2816** sem bio — diferente do
braço VideoMAE (2688/2560) porque `CLIP_VIDEO_DIM=1024` substitui os 768 do VideoMAE.

In [ ]:
model = vcc.build_model_clip(INIT_SEED, use_bio=USE_BIO,
                             use_viewer_emb=USE_VIEWER_EMB)

expected_fusion_in = vcc.fusion_input_dim_clip(USE_BIO)
assert model.fusion_in == expected_fusion_in, (
    f"fusion_in={model.fusion_in} != esperado {expected_fusion_in} "
    f"(2944 com bio / 2816 sem bio)")
print("fusion_in:", model.fusion_in,
      "(2944 = com slot bio | 2816 = sem slot bio)")
print("params:", vc.count_params(model))
vc.record_fingerprint(RUN_DIR, "init", model)


## 8.1 Smoke test — PORTÃO DURO (célula 19b)

⚠ **Nada abaixo desta célula deve rodar na VM sem que a célula seguinte passe
primeiro.** `vcc.smoke_test_clip` roda 1 forward + backward + step sob bf16 e gradient
checkpointing, mede memória de pico, extrapola o timing para 1 época inteira, e
verifica que os gradientes das 4 camadas destravadas do CLIP não são nulos.

Isso existe porque este braço tem três riscos pré-declarados que só se manifestam em
runtime real (`improvements/18_CLIP_VIDEO_BRANCH_SPEC.md`, riscos R1/R3/R4):
OOM (32 imagens ViT-L por forward: 16 frames × micro-bs 2, mais grad em 4 camadas),
o caminho de atributo do unfreeze não bater na versão de `transformers` da VM, e uma
época bem mais lenta que o braço VideoMAE. Rodar as 15 épocas sem passar por este portão
arrisca descobrir qualquer um dos três só depois de horas de GPU.

In [ ]:
vcc.smoke_test_clip(model, train_dataset, device="cuda", micro_bs=2)
print("✓ smoke_test_clip passou — seguro prosseguir para o treino completo")


## 9. `TrainingArguments`

**Idêntica à congelada de 15 épocas do braço VideoMAE** (bs 2, accum 16, bf16,
gradient checkpointing) — mantida assim por comparabilidade de custo/orçamento com
B15, não porque o braço CLIP tenha sido calibrado com esses valores. Qualquer alteração
aqui invalida a leitura de timing feita no smoke test da célula 19b.

- `warmup_steps=0.1` (float `[0,1)` = ratio) — mesma durabilidade documentada no
  notebook VideoMAE (`warmup_ratio` funciona na VM, `transformers==5.6.0`, mas está
  deprecado).
- `load_best_model_at_end=False`, `metric_for_best_model` deliberadamente ausente —
  mesmo motivo do braço VideoMAE (selecionar época pela métrica reportada é leakage).
- **`save_strategy="no"`** — o modelo final é salvo à mão, enxuto, na seção 13
  (`vcc.save_run_clip`), no estado FINAL da época 15 (decisão final, ver célula 0/2.1).
  A seção 10 ainda registra um `SaveTrainableStateCallback` que grava o estado
  treinável por época em `epoch_states/` — **só como registro diagnóstico** (permite
  reconstruir a curva de qualquer época sem re-treinar, útil se alguém quiser
  investigar de novo), a seção 12 já não recarrega nada dali por cima do modelo final.
  `epoch_states/` continua sendo removido (seção 14) depois que a run oficial é
  validada.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=NUM_EPOCHS,

    # ── batch: IDÊNTICA à congelada do braço VideoMAE (comparabilidade de custo) ──
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,          # batch efetivo = 32
    gradient_checkpointing=True,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    dataloader_persistent_workers=True,
    dataloader_pin_memory=True,

    eval_strategy="epoch",
    save_strategy="no",                      # checkpoint completo do HF é caro demais
    # (backbone congelado + otimizador). Estado por época salvo enxuto via callback
    # (seção 10) — ver célula markdown acima.
    load_best_model_at_end=vc.LOAD_BEST_MODEL_AT_END,   # False — não ligar (leakage)
    # metric_for_best_model: deliberadamente AUSENTE. Ver markdown acima.

    warmup_steps=0.1,                        # float ∈ [0,1) = ratio (durabilidade, não bugfix)
    lr_scheduler_type="cosine",

    remove_unused_columns=False,
    label_names=["labels_a", "labels_b", "labels_c"],

    report_to="none",
    logging_steps=10,
    seed=INIT_SEED,
    data_seed=INIT_SEED,
)


## 10. Trainer

`data_collator=vc.video_collate` é **obrigatório**, mesmo motivo do braço VideoMAE:
o dataset devolve `id_EXIST` (str).

O otimizador é `vcc.get_optimizer_clip(model)` — não o `vc.get_optimizer(model)` default.
Tem um grupo dedicado para os parâmetros `videomae.*` (que aqui são os do CLIP, wart de
nomenclatura documentada) com `lr=CLIP_LR=1e-5`, verificado **primeiro** no predicado
antes de cair no `base_decay` (1e-3) — cair lá destruiria os pesos pré-treinados do
ViT-L sem erro nenhum. `get_optimizer_clip` já roda dois asserts internos (grupo CLIP
com a contagem esperada de params p/ `UNFROZEN_VISION_LAYERS`; partição exata) antes de
devolver o otimizador.

**`SaveTrainableStateCallback` (registro diagnóstico, não usado pra seleção):** salva
`vc.trainable_state_dict(model)` — só os params treináveis — em
`CKPT_DIR/epoch_states/epoch_NN.safetensors` a cada `on_evaluate`. A run oficial é o
estado FINAL da época 15 (decisão da célula 0/2.1); estes arquivos existem só para
permitir reconstruir a curva de qualquer época sem re-treinar, e são removidos na
seção 14 depois que a run oficial é validada.

`trainer.label_names` é setado também depois da construção — padrão idêntico ao braço
VideoMAE, copiado literalmente (é a forma empiricamente conhecida por funcionar em
`transformers==5.6.0`).

In [ ]:
from transformers import TrainerCallback
from safetensors.torch import save_file as st_save_file
from accelerate.utils import extract_model_from_parallel

EPOCH_STATES_DIR = CKPT_DIR / "epoch_states"

class SaveTrainableStateCallback(TrainerCallback):
    """Salva vc.trainable_state_dict(model) — só os params TREINÁVEIS, ~220MB — a
    cada avaliação (dispara junto com eval_strategy="epoch", 1x/época). Substitui o
    save_strategy="epoch" do HF Trainer, que salvaria o checkpoint completo (backbone
    congelado + otimizador) e arriscaria esgotar disco em 15 épocas — ver célula
    markdown da seção 9."""

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        m = extract_model_from_parallel(model if model is not None else kwargs["model"])
        epoch_int = round(state.epoch)
        EPOCH_STATES_DIR.mkdir(parents=True, exist_ok=True)
        path = EPOCH_STATES_DIR / f"epoch_{epoch_int:02d}.safetensors"
        st_save_file(vc.trainable_state_dict(m), str(path))
        print(f"[epoch_states] época {epoch_int} salva em {path.name}")

epoch_state_callback = SaveTrainableStateCallback()

optimizer = vcc.get_optimizer_clip(model)

trainer = vc.MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=vc.compute_metrics,
    data_collator=vc.video_collate,
    optimizers=(optimizer, None),
    callbacks=[epoch_state_callback],
)
trainer.label_names = ["labels_a", "labels_b", "labels_c"]

print("model_accepts_loss_kwargs:", trainer.model_accepts_loss_kwargs, "(travado em False)")
total_steps = (len(train_dataset) // (training_args.per_device_train_batch_size
               * training_args.gradient_accumulation_steps)) * NUM_EPOCHS
print("warmup steps:", training_args.get_warmup_steps(total_steps),
      f"de ~{total_steps} steps totais")


## 11. Treino

In [ ]:
# Guarda contra execução fora de ordem do notebook: se a célula 2.2 não rodou, este
# treino usaria LOSS_W_C=0.7 (o baseline) e só se descobriria comparando métricas.
assert vc.LOSS_W_C == vcc.LOSS_W_C_REWEIGHTED, (
    f"LOSS_W_C={vc.LOSS_W_C} != {vcc.LOSS_W_C_REWEIGHTED} — rodar a célula 2.2 "
    f"(vcc.apply_loss_reweight_c()) ANTES de treinar")
print(f"[pré-treino] LOSS_W_A={vc.LOSS_W_A} LOSS_W_B={vc.LOSS_W_B} LOSS_W_C={vc.LOSS_W_C}")

train_output = trainer.train()
print(train_output)

In [ ]:
# Por que FMeasure/CrossEntropy vêm NaN nas tasks B e C — mesmo comportamento
# documentado no braço VideoMAE, não é específico do encoder CLIP.
import numpy as np, tempfile
from pyevall.evaluation import PyEvALLEvaluation
import video_core as vc

pred_out = trainer.predict(val_dataset)
_, logits_b, _ = vc._unpack_predictions(pred_out.predictions)
_, labels_b, _ = pred_out.label_ids
logits_b, labels_b = np.asarray(logits_b), np.asarray(labels_b)
mask = labels_b[:, 0] != vc.SENTINEL_BC
x = logits_b[mask]; x = x - x.max(axis=-1, keepdims=True)
probs = np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
ids = [str(i) for i in range(len(probs))]

with tempfile.TemporaryDirectory() as td:
    ml = vc._metric_list('2_2', 'hard')
    preds = vc._format_for_pyevall(probs, ids, '2_2', 'hard')
    gold = vc._format_for_pyevall(labels_b[mask], ids, '2_2', 'hard')
    import json, os
    pp, gp = f"{td}/p.json", f"{td}/g.json"
    json.dump(preds, open(pp, "w")); json.dump(gold, open(gp, "w"))
    from pyevall.utils.utils import PyEvALLUtils
    rep = PyEvALLEvaluation().evaluate(
        pp, gp, ml, **{PyEvALLUtils.PARAM_HIERARCHY: vc.TASK_2_2_HIERARCHY})
    for m in ml:
        d = rep.report.get('metrics', {}).get(m)
        print(f"  {m}: {'AUSENTE do relatório' if d is None else 'status=' + str(d.get('status'))}")


## 12. Curva de validação (registro) — estado oficial é o FINAL, não argmin

Histórico do que foi tentado e revertido, pra quem reabrir este notebook não repetir o
caminho: uma 2ª run recarregou `E=argmin(eval_loss)` (época 2) por cima do modelo final,
testando a hipótese de que overtraining explicava o ramo de texto ficando quase
constante na fusão. Uma 3ª run também testou `UNFROZEN_VISION_LAYERS` 4→2 (hipótese de
desequilíbrio de capacidade do encoder). As duas hipóteses foram **refutadas** pela
sonda causal (`probe_video_contribution_clip.ipynb`): em nenhuma variante o slot de
texto recuperou distância do baseline VideoMAE, e a 3ª run piorou (vídeo dominou
*mais* com menos capacidade). Detalhes e números em
`improvements/16_VIDEO_INVESTIGATION_RESULTS.md`.

Conclusão: o padrão é um trade-off real de trocar VideoMAE por CLIP, não um artefato de
treino corrigível por seleção de época ou capacidade do encoder. Este braço volta a ser
reportado no **estado FINAL da época 15** — a célula abaixo só imprime a curva de
`eval_loss`/métricas por época **para registro/diagnóstico**; `actual_model` continua
sendo o modelo exatamente como o treino o deixou (nenhum reload acontece).

In [ ]:
evals = [h for h in trainer.state.log_history if "eval_loss" in h]
assert evals, "nenhum eval_loss no log_history — eval_strategy deve estar 'epoch'"

curva = pd.DataFrame(evals)
cols = [c for c in curva.columns
        if c in ("epoch", "step", "eval_loss")
        or c.startswith(("eval_task_a", "eval_task_b", "eval_task_c"))]
print(curva[cols].to_string(index=False))

best = min(evals, key=lambda h: h["eval_loss"])
best_epoch_int = round(best["epoch"])
print(f"\nmin(eval_loss) na época {best_epoch_int} (eval_loss = {best['eval_loss']:.4f}) "
      f"— REGISTRO APENAS. Este braço reporta o estado FINAL da época {NUM_EPOCHS} "
      f"(decisão da célula 0/2.1, ver histórico ali) — nenhum reload acontece aqui.")

# actual_model = exatamente o que o treino deixou (época NUM_EPOCHS), sem
# recarregar nada de epoch_states/. extract_model_from_parallel já foi importado
# na seção 10.
actual_model = extract_model_from_parallel(trainer.model)
print(f"✓ actual_model = estado FINAL da época {NUM_EPOCHS} — este é o estado que "
      f"a seção 13 salva")


## 13. Salvar a run (checkpoint enxuto, estado FINAL da época 15)

`vcc.save_run_clip` grava um `meta.json` **novo**, diferente do braço VideoMAE:
`video_encoder`, `unfrozen_vision_layers`, `video_hidden_dim`, `n_frames`,
`frame_sampling`, `clip_image_mean`/`_std`, `clip_lr` — **sem** `videomae_qv_biases`/
`restore_biases` (não se aplicam a este encoder). `video_encoder` é o desambiguador
autoritativo contra o wart de nomenclatura (`videomae.clip.*` no `state_dict`).

`actual_model` é o estado FINAL da época `NUM_EPOCHS` (célula 12 não recarrega nada) —
o `meta` abaixo registra `epoch_selection="final"` e `min_eval_loss_epoch` (o argmin,
só como referência histórica) para deixar a decisão rastreável no artefato salvo.

In [ ]:
vcc.save_run_clip(actual_model, RUN_DIR, scaler=scaler, tokenizer=tokenizer,
                   meta={"init_seed": INIT_SEED, "condition": CONDITION,
                         "num_train_epochs": NUM_EPOCHS,
                         "selected_epoch": NUM_EPOCHS,
                         "epoch_selection": "final",
                         "min_eval_loss_epoch": best_epoch_int,
                         "min_eval_loss_value": float(best["eval_loss"]),
                         "train_runtime_s": train_output.metrics.get("train_runtime"),
                         "arm": ARM,
                         **LOSS_WEIGHTS,
                         "audio_version": AUDIO_VERSION,
                         "frames_version": vc.FEATURES_VERSION})
vc.record_fingerprint(RUN_DIR, "final", actual_model)


## 14. Validação do checkpoint

Mesmo padrão do braço VideoMAE: lê exatamente o que a gravação escreveu (via
`vcc.load_run_clip`, que assert-a `meta["video_encoder"]` e `fusion_in ==
fusion_input_dim_clip(...)`) e falha alto se as chaves não baterem.

In [ ]:
model_check, scaler_check = vcc.load_run_clip(RUN_DIR, init_seed=INIT_SEED)

mismatches = 0
for key in ["fusion.0.weight", "head_a.weight", "head_b.weight", "head_c.weight"]:
    orig = actual_model.state_dict()[key].cpu()
    loaded = model_check.state_dict()[key].cpu()
    if torch.allclose(orig, loaded):
        print(f"  OK: {key}")
    else:
        print(f"  MISMATCH: {key}")
        mismatches += 1

fresh = vcc.build_model_clip(INIT_SEED, use_bio=USE_BIO, use_viewer_emb=USE_VIEWER_EMB)
differs_from_fresh = not torch.allclose(
    model_check.state_dict()["head_a.weight"].cpu(),
    fresh.state_dict()["head_a.weight"].cpu())

if mismatches == 0 and differs_from_fresh:
    print("\n✓ Checkpoint válido — bate com o modelo treinado e difere do init aleatório")
elif mismatches:
    print(f"\n✗ {mismatches} divergências de peso — algo correu mal no salvamento")
else:
    print("\n✗ Os pesos carregados são idênticos ao init aleatório — não foi salvo")

del model_check, fresh

# Limpeza dos estados por época (seção 10/12) — só depois de RUN_DIR confirmado válido
# acima. Cada arquivo é ~220MB × 15 ≈ 3,3GB; servem só como fonte para o load da
# seção 12, já não têm função depois que a run oficial está gravada e verificada.
if mismatches == 0 and differs_from_fresh:
    import shutil
    n_removed = len(list(EPOCH_STATES_DIR.glob("*.safetensors")))
    shutil.rmtree(EPOCH_STATES_DIR, ignore_errors=True)
    print(f"✓ {n_removed} arquivos de epoch_states/ removidos (~{n_removed * 220}MB liberados)")
else:
    print("⚠ epoch_states/ NÃO removido — checkpoint não passou na validação acima, "
          "mantido para investigação")


## 15. Ajuste de limiares da Task C (uma vez, pós-treino)

Mesmo protocolo do braço VideoMAE — limiares ajustados numa metade do val, número
reportável vem da outra metade (ARCH-11 b). `NONE` fixo em 0,5.

In [ ]:
threshold_result = vc.tune_task_c_thresholds(
    trainer, val_dataset,
    save_path=str(RUN_DIR / "task_c_thresholds.json"),
)


## 16. Avaliação final e próximos passos

`trainer.evaluate()` é o caminho único de avaliação, mesmo motivo do braço VideoMAE
(OPT-08). `trainer.model` não foi mutado (célula 12 não recarrega nada) — `final_metrics`
abaixo reflete o estado FINAL da época `NUM_EPOCHS`, a run oficial deste braço.

**Depois desta run:** rodar `probe_video_contribution_clip.ipynb` contra este `RUN_DIR`
(mesma metodologia, mesmos slots — o `CACHE_DIR` auto-deriva do nome da run, então o
cache novo sai de graça). ⚠ Aqui o objetivo é **MEDIR** o comportamento do ramo de texto
sob a loss repesada, **não confirmar** os números da run `CLIP` (`‖Δ_texto‖ = 0,785`):
reproduzir 0,785 não é esperado nem é critério de sucesso, e um Δ diferente **não** é
falha — é o resultado. As duas hipóteses de correção do colapso do texto (argmin cedo e
capacidade do encoder) já foram testadas e refutadas em runs anteriores; não repetir.

⚠ Comparar `eval_loss` desta run com a de qualquer run anterior é INVÁLIDO (o termo de C
entra 6× maior). Só as 6 métricas do PyEvALL comparam — e as seis, não só a Task C:
esta intervenção é realocação, subir C deve custar A e/ou B.

Os números deste braço são reportáveis apenas como atribuição isolada (troca de
encoder), **não** entram na tabela de ablação cross-seed do sistema VideoMAE (que é
pareada por seed, com o mesmo encoder em todos os braços). Comparar com B15 é uma
leitura qualitativa de teto, sujeita à limitação de interpolação declarada na célula 0 e
em `improvements/18_CLIP_VIDEO_BRANCH_SPEC.md` — nunca uma comparação controlada.

In [ ]:
final_metrics = trainer.evaluate()
for k, v in sorted(final_metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

with open(RUN_DIR / "final_metrics.json", "w") as f:
    json.dump({k: (float(v) if isinstance(v, (int, float)) else v)
               for k, v in final_metrics.items()}, f, indent=2)
print(f"\n✓ métricas gravadas em {RUN_DIR / 'final_metrics.json'}")
